In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# ensure deterministic dataset batching
tf.config.experimental.enable_op_determinism()

Loading, Cleaning and Exploring the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

path = "/content/SPX500-18Y_scaled_y.csv"
data = pd.read_csv(path, parse_dates=['observation_date'], dayfirst=True)

print(data.shape)
data = data.dropna(axis=1, how='all')
data = data.drop(index=0)

display(data.head())

print(data.info())
print(data.isna().sum())

display(data.describe().T)


(4789, 49)


,observation_date,Open,High,Low,Close,Adj Close,Volume
1,2004-01-02,1111.920044,1118.849976,1105.079956,1108.479980,1108.479980,1.153200e+09
2,2004-01-05,1108.479980,1122.219971,1108.479980,1122.219971,1122.219971,1.578200e+09
3,2004-01-06,1122.219971,1124.459961,1118.439941,1123.670044,1123.670044,1.494500e+09
4,2004-01-07,1123.670044,1126.329956,1116.449951,1126.329956,1126.329956,1.704900e+09
5,2004-01-08,1126.329956,1131.920044,1124.910034,1131.920044,1131.920044,1.868400e+09


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4788 entries, 1 to 4788
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   observation_date  4783 non-null   datetime64[ns]
 1   Open              4783 non-null   float64       
 2   High              4783 non-null   float64       
 3   Low               4783 non-null   float64       
 4   Close             4783 non-null   float64       
 5   Adj Close         4783 non-null   float64       
 6   Volume            4783 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 262.0 KB
None
observation_date    5
Open                5
High                5
Low                 5
Close               5
Adj Close           5
Volume              5
dtype: int64


,count,mean,min,25%,50%,75%,max,std
observation_date,4783,2013-07-02 10:36:45.310474752,2004-01-02 00:00:00,2008-10-01 12:00:00,2013-07-03 00:00:00,2018-04-03 12:00:00,2022-12-30 00:00:00,NaN
Open,4783.0,2026.499451,679.280029,1257.690002,1649.140015,2657.400024,4804.509766,992.156431
High,4783.0,2038.047512,695.27002,1264.390015,1656.569946,2674.75,4818.620117,997.605489
Low,4783.0,2014.028137,666.789978,1249.61499,1639.77002,2638.599976,4780.040039,986.021098
Close,4783.0,2026.779142,676.530029,1257.845032,1650.469971,2656.585083,4796.560059,992.106721
Adj Close,4783.0,2026.779142,676.530029,1257.845032,1650.469971,2656.585083,4796.560059,992.106721
Volume,4783.0,3723250771.482333,504580000.0,3005670000.0,3641230000.0,4342135000.0,11456230000.0,1324765065.435597


Feature Engineering

## Feature Engineering — Improvements

Three changes made to the original feature set:

**1. Raw OHLC → log-ratio features**  
Raw prices are non-stationary (700 in 2004 → 4800 in 2022). Replaced with `log(today/yesterday_close)` for Open, High, Low, Close. These are scale-free percentage moves the model can compare across all 18 years.

**2. Volume lags added**  
The original model had no lagged volume, so it could not tell if volume was trending up or down. Added `log_volume_lag1` and `log_volume_lag5`.

**3. Rolling skewness added**  
A 20-day rolling skew of returns signals whether the return distribution is tilting toward the left tail — directly relevant for VaR and CVaR prediction.


In [ ]:
# --- Basic return and volume features ---
data['ret'] = data['Adj Close'].pct_change(fill_method=None)
data['log_return'] = np.log(data['Adj Close'] / data['Adj Close'].shift(1))
data['log_volume'] = np.log(data['Volume'])

# --- FIX 1: Replace raw OHLC with log-ratio features (stationary, scale-free) ---
# Raw prices (e.g. 700 in 2004 vs 4800 in 2022) are non-stationary and mislead the model.
# log(today / yesterday) gives a normalised percentage move the model can compare across all years.
data['log_open_ret']  = np.log(data['Open']  / data['Adj Close'].shift(1))
data['log_high_ret']  = np.log(data['High']  / data['Adj Close'].shift(1))
data['log_low_ret']   = np.log(data['Low']   / data['Adj Close'].shift(1))
data['log_close_ret'] = np.log(data['Close'] / data['Adj Close'].shift(1))

# --- FIX 2: Volume lags so model knows if volume is trending up or down ---
data['log_volume_lag1'] = data['log_volume'].shift(1)
data['log_volume_lag5'] = data['log_volume'].shift(5)

# --- FIX 3: Rolling skewness — signals whether tail risk is skewing negative ---
data['ret_skew20'] = data['log_return'].rolling(window=20).skew()

data.dropna(inplace=True)


In [ ]:
def yang_zhang_vol(df):
    log_oc = np.log(df['Open'] / df['Adj Close'].shift(1))
    log_ho = np.log(df['High'] / df['Open'])
    log_lo = np.log(df['Low'] / df['Open'])
    rs = 0.5 * (log_ho**2 + log_lo**2)
    return (log_oc**2 + rs).rolling(window=20).mean()

data['yz_vol'] = yang_zhang_vol(data)
data.dropna(inplace=True)

In [ ]:
# Compute a volatility-adjusted 5% VaR (parametric approximation)
data['VaR_5'] = data['log_return'].ewm(span=10).mean() - 1.2 * data['yz_vol']
data.dropna(inplace=True)


In [ ]:
#data['log_volume_smooth'] = data['log_volume'].rolling(window=10).mean()

for lag in range(1, 6):
    data[f'ret_lag{lag}'] = data['log_return'].shift(lag)

data['yz_vol_lag1'] = data['yz_vol'].shift(1)
data['VaR_5_lag1'] = data['VaR_5'].shift(1)

# Expected shortfall (CVaR) for 5% quantile
data['CVaR_5'] = data['log_return'].rolling(window=20).apply(lambda x: x[x < x.quantile(0.05)].mean())

data.dropna(inplace=True)


In [ ]:
data['drawdown'] = (data['Adj Close'] / data['Adj Close'].cummax()) - 1
data['ret_min20'] = data['log_return'].rolling(20).min()
data['ret_max20'] = data['log_return'].rolling(20).max()
data['CVaR_5_lag1'] = data['CVaR_5'].shift(1)
data.dropna(inplace=True)


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Updated feature set:
# - log_open/high/low/close_ret replace raw OHLC (normalised, stationary)
# - log_volume_lag1/5 added (volume trend context)
# - ret_skew20 added (tail direction signal)
features = [
    'log_open_ret', 'log_high_ret', 'log_low_ret', 'log_close_ret',
    'log_volume_lag1', 'log_volume_lag5',
    'ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5',
    'yz_vol_lag1', 'VaR_5_lag1', 'drawdown', 'ret_min20', 'ret_max20',
    'CVaR_5_lag1', 'ret_skew20'
]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[features])

# Create 5-day rolling sequences
sequence_length = 5
X_seq = np.array([
    X_scaled[i-sequence_length+1:i+1]
    for i in range(sequence_length - 1, len(X_scaled))
])  # shape: (n_samples, 5, n_features)

# Align targets with sequences
y_return = data['log_return'].values[sequence_length - 1:]
y_volume = data['log_volume'].values[sequence_length - 1:]
y_yz     = data['yz_vol'].values[sequence_length - 1:]
y_var    = data['VaR_5'].values[sequence_length - 1:]
y_cvar   = data['CVaR_5'].values[sequence_length - 1:]

# Train-test split — preserving temporal order (no shuffling)
split_index = int(len(X_seq) * 0.8)

X_train, X_test = X_seq[:split_index], X_seq[split_index:]

y_train_r,   y_test_r   = y_return[:split_index], y_return[split_index:]
y_train_v,   y_test_v   = y_volume[:split_index], y_volume[split_index:]
y_train_yz,  y_test_yz  = y_yz[:split_index],     y_yz[split_index:]
y_train_var, y_test_var = y_var[:split_index],     y_var[split_index:]
y_train_cvar,y_test_cvar= y_cvar[:split_index],    y_cvar[split_index:]

print(f"Features: {len(features)}")
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Input shape: {X_train.shape}")


Modelling

to control randomness ensuring we get 1 result everytime we run the code

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
tf.random.set_seed(42)


Model Architecture: Multi-output neural network model using TensorFlow/Keras for financial time series prediction. It first defines the input shape as a sequence of sequence_length days with n_features per day. The input is flattened and passed through two fully connected layers with 256 and 128 neurons and ReLU activation to learn high-level patterns. The model then has five separate output layers, each predicting a different financial target: log_return, log_volume, yz_vol (volatility), VaR_5 (Value at Risk), and CVaR_5 (Conditional Value at Risk).

In [ ]:
from tensorflow.keras import layers, Model, Input

sequence_length = 5
n_features = len(features)

def build_concept_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    x = layers.Flatten()(inp)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    out_mu   = layers.Dense(1, name="log_return")(x)
    out_vol  = layers.Dense(1, name="log_volume")(x)
    out_yz   = layers.Dense(1, name="yz_vol")(x)
    out_var  = layers.Dense(1, name="VaR_5")(x)
    out_cvar = layers.Dense(1, name="CVaR_5")(x)
    return Model(inp, [out_mu, out_vol, out_yz, out_var, out_cvar])

concept_model = build_concept_model(sequence_length, n_features)
concept_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 5, 16)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_8 (Flatten) │ (None, 80)        │          0 │ input_layer_13[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_61 (Dense)    │ (None, 256)       │     20,736 │ flatten_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_62 (Dense)    │ (None, 128)       │     32,896 │ dense_61[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ log_return (Dense)  │ (None, 1)         │        129 │ dense_62[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ log_volume (Dense)  │ (None, 1)         │        129 │ dense_62[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ yz_vol (Dense)      │ (None, 1)         │        129 │ dense_62[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ VaR_5 (Dense)       │ (None, 1)         │        129 │ dense_62[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CVaR_5 (Dense)      │ (None, 1)         │        129 │ dense_62[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,277 (212.02 KB)

 Trainable params: 54,277 (212.02 KB)

 Non-trainable params: 0 (0.00 B)

We use different loss functions:
Tail-Weighted Loss (Quantiles): For VaR loss
Huber Loss(MAE + MSE), Robust to Outliers: For log_return, volume and y_volatiltiy
Log_Cosh : For CvAR (Less sensitive to extreme outliers behaves as MSE for small errors.

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Huber loss
@tf.keras.utils.register_keras_serializable()
def huber_loss(y_true, y_pred, delta=1.0):
    e = y_true - y_pred
    return tf.reduce_mean(tf.where(tf.abs(e) < delta,
                                   0.5 * e**2,
                                   delta * (tf.abs(e) - 0.5 * delta)))

# Log-Cosh loss
@tf.keras.utils.register_keras_serializable()
def log_cosh(y_true, y_pred):
    return tf.reduce_mean(tf.math.log(tf.cosh(y_pred - y_true + 1e-12)))

# Quantile loss for 5% (VaR)
@tf.keras.utils.register_keras_serializable()
def quantile_loss_05(y_true, y_pred):
    q = 0.05
    e = y_true - y_pred
    return tf.reduce_mean(tf.maximum(q * e, (q - 1) * e))



Each output is assigned a custom loss function that matches its statistical behavior: Huber loss for returns, volume, and volatility to handle occasional spikes; quantile loss for VaR to focus on the 5% lower tail; and log-cosh for CVaR to be robust to extreme tail events. The model is trained for 5 epochs with a batch size of 32, using validation data to monitor performance.

In [ ]:
# FIX: EarlyStopping must be defined and passed BEFORE fit() — previously it was created after,
# so it never fired and the model trained all 50 epochs regardless of validation loss.
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

concept_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={
        "log_return": huber_loss,
        "log_volume": huber_loss,
        "yz_vol":     huber_loss,
        "VaR_5":      quantile_loss_05,
        "CVaR_5":     log_cosh,
    }
)

concept_model.fit(
    X_train,
    [y_train_r, y_train_v, y_train_yz, y_train_var, y_train_cvar],
    validation_data=(
        X_test,
        [y_test_r, y_test_v, y_test_yz, y_test_var, y_test_cvar]
    ),
    epochs=50,
    batch_size=32,
    callbacks=[callback]   # now actually connected
)

concept_model.save("concept_model.keras")


Epoch 1/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - CVaR_5_loss: 0.0314 - VaR_5_loss: 0.0572 - log_return_loss: 0.0249 - log_volume_loss: 10.8393 - loss: 10.9982 - yz_vol_loss: 0.0455 - val_CVaR_5_loss: 0.0465 - val_VaR_5_loss: 0.0419 - val_log_return_loss: 0.0164 - val_log_volume_loss: 10.7009 - val_loss: 10.9876 - val_yz_vol_loss: 0.0493
Epoch 2/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - CVaR_5_loss: 0.0111 - VaR_5_loss: 0.0162 - log_return_loss: 0.0077 - log_volume_loss: 1.5078 - loss: 1.5554 - yz_vol_loss: 0.0126 - val_CVaR_5_loss: 0.0122 - val_VaR_5_loss: 0.0448 - val_log_return_loss: 0.0150 - val_log_volume_loss: 9.4039 - val_loss: 9.5849 - val_yz_vol_loss: 0.0255
Epoch 3/50
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - CVaR_5_loss: 0.0056 - VaR_5_loss: 0.0169 - log_return_loss: 0.0049 - log_volume_loss: 0.9819 - loss: 1.0161 - yz_vol_loss: 0.0068 - val_CVaR_5_loss: 0.0085 - val_VaR_5_loss: 0.0375 - val_log_return_loss: 0.0120 - val_log_volume_loss: 9.6509 - val_loss: 9.7727 - 

This code evaluates the trained model’s performance on the validation set by computing the average prediction error for each output using the same loss functions applied during training, but implemented in NumPy for easy computation. First, the model predicts all five targets log return, log volume, volatility, VaR, and CVaR on the validation inputs, and each prediction is unpacked into separate arrays. Then, NumPy versions of Huber loss, quantile loss, and log-cosh loss are defined to measure the discrepancy between predicted and true values. Finally, the code calculates the mean loss for each output and stores them in the loss_means dictionary.

In [ ]:
preds = concept_model.predict(X_test)
mu_p, vol_p, yz_p, var_p, cvar_p = preds

def huber_np(y, p, delta=1.0):
    e = y - p
    return np.mean(np.where(np.abs(e) < delta, 0.5*e**2, delta*(np.abs(e)-0.5*delta)))

def quantile_np(y, p, q=0.05):
    e = y - p
    return np.mean(np.maximum(q*e, (q-1)*e))

def log_cosh_np(y, p):
    return np.mean(np.log(np.cosh(p - y)))

loss_means = {
    "log_return": huber_np(y_test_r, mu_p),
    "log_volume": huber_np(y_test_v, vol_p),
    "yz_vol":     huber_np(y_test_yz, yz_p),
    "VaR_5":      quantile_np(y_test_var, var_p),
    "CVaR_5":     log_cosh_np(y_test_cvar, cvar_p),
}

loss_means

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


{'log_return': np.float64(0.015477943919500636),
 'log_volume': np.float64(1.751546564278033),
 'yz_vol': np.float64(0.016337551448293828),
 'VaR_5': np.float64(0.031971194987086025),
 'CVaR_5': np.float64(0.013768654124669693)}

more weight to outputs with lower validation loss

In [ ]:
inv = {k: 1.0/(v + 1e-12) for k, v in loss_means.items()}
s = sum(inv.values())
weights = {k: inv[k]/s for k in inv}

weights

{'log_return': np.float64(0.2805453276413959),
 'log_volume': np.float64(0.0024791032890292323),
 'yz_vol': np.float64(0.2657843105724201),
 'VaR_5': np.float64(0.13581803401499803),
 'CVaR_5': np.float64(0.3153732244821568)}

Takes sequences of past financial features as input, passes them through a dense projection layer, and then through four stacked Transformer blocks that use multi-head attention to capture dependencies across timesteps. After aggregating the sequence with global average pooling, the model produces five separate outputs—log return, log volume, volatility, VaR, and CVaR—using dedicated dense layers.

In [ ]:
def transformer_block(x, heads=4, ff_dim=128):
    attn = layers.MultiHeadAttention(num_heads=heads, key_dim=ff_dim)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    f = layers.Dense(ff_dim, activation="relu")(x)
    f = layers.Dense(x.shape[-1])(f)
    x = layers.Add()([x, f])
    x = layers.LayerNormalization()(x)
    return x

def build_transformer_model(seq_len, n_features):
    inp = Input(shape=(seq_len, n_features))
    x = layers.Dense(64, activation=None)(inp)

    for _ in range(4):
        x = transformer_block(x)

    x = layers.GlobalAveragePooling1D()(x)

    out_mu   = layers.Dense(1, name="log_return")(x)
    out_vol  = layers.Dense(1, name="log_volume")(x)
    out_yz   = layers.Dense(1, name="yz_vol")(x)
    out_var  = layers.Dense(1, name="VaR_5")(x)
    out_cvar = layers.Dense(1, name="CVaR_5")(x)

    return Model(inp, [out_mu, out_vol, out_yz, out_var, out_cvar])

trans_model = build_transformer_model(sequence_length, n_features)

Performs weight transfer from a previously trained fully-connected model to a new Transformer-based model. First, it loads the saved concept_model using custom_objects={"loss": huber_loss} so that Keras can recognize the custom Huber loss function. Then, it iterates over the first five layers of both models, and for each pair of compatible layers, it copies the weights from the old model to the new one and freezes them (trainable = False) to prevent further training. Layers whose weights don’t match in shape are skipped.

In [ ]:
concept_loaded = tf.keras.models.load_model(
    "concept_model.keras",
    custom_objects={
        "huber_loss": huber_loss,
        "log_cosh": log_cosh,
        "quantile_loss_05": quantile_loss_05
    }
)

# transfer compatible weights
for lt, lc in zip(trans_model.layers[:5], concept_loaded.layers[:5]):
    try:
        lt.set_weights(lc.get_weights())
        lt.trainable = False
        print("Transferred:", lt.name)
    except:
        print("Skipped:", lt.name)

Transferred: input_layer_14
Skipped: dense_63
Skipped: multi_head_attention_20
Skipped: add_40
Skipped: layer_normalization_40


defines a weighted loss function for multi-output training, where each task’s individual loss is multiplied by a weight reflecting its predictive reliability or importance. For each output, the appropriate loss function is applied—Huber for returns, volume, and volatility; quantile loss for VaR; and log-cosh for CVaR. By scaling each loss with its corresponding weight, the model ensures balanced learning across tasks.

In [ ]:
def weighted_loss(y_true, y_pred, task):
    w = weights[task]

    if task in ["log_return", "log_volume", "yz_vol"]:
        return w * huber_loss(y_true, y_pred)

    if task == "VaR_5":
        return w * quantile_loss_05(y_true, y_pred)

    if task == "CVaR_5":
        return w * log_cosh(y_true, y_pred)


This code trains the Transformer model step by step for all five outputs at once. It breaks the training data into batches and goes through them for several epochs. For each batch, the model makes predictions, computes a weighted sum of losses for all outputs, calculates gradients showing how to adjust each weight to reduce loss, and updates the model weights using the Adam optimizer. After each epoch, it prints the loss of the last batch to track progress. This manual approach gives more control over how multiple outputs and their weights are handled, which is harder to do using model.fit.

In [ ]:
optimizer = tf.keras.optimizers.Adam(1e-4)
epochs = 50
batch_size = 32

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train,
     {'log_return': tf.cast(y_train_r,   tf.float32),
      'log_volume': tf.cast(y_train_v,   tf.float32),
      'yz_vol':     tf.cast(y_train_yz,  tf.float32),
      'VaR_5':      tf.cast(y_train_var, tf.float32),
      'CVaR_5':     tf.cast(y_train_cvar,tf.float32)})).batch(batch_size)

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_test,
     {'log_return': tf.cast(y_test_r,   tf.float32),
      'log_volume': tf.cast(y_test_v,   tf.float32),
      'yz_vol':     tf.cast(y_test_yz,  tf.float32),
      'VaR_5':      tf.cast(y_test_var, tf.float32),
      'CVaR_5':     tf.cast(y_test_cvar,tf.float32)})).batch(batch_size)

best_val_loss = float('inf')
patience, wait = 5, 0

for epoch in range(epochs):
    # --- Training pass ---
    for xb, yb in train_ds:
        with tf.GradientTape() as tape:
            mu_p, vol_p, yz_p, var_p, cvar_p = trans_model(xb, training=True)
            L = (
                weighted_loss(yb["log_return"], mu_p,  "log_return") +
                weighted_loss(yb["log_volume"], vol_p, "log_volume") +
                weighted_loss(yb["yz_vol"],     yz_p,  "yz_vol")    +
                weighted_loss(yb["VaR_5"],      var_p, "VaR_5")     +
                weighted_loss(yb["CVaR_5"],     cvar_p,"CVaR_5")
            )
        grads = tape.gradient(L, trans_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, trans_model.trainable_variables))

    # --- Validation pass (added) ---
    val_losses = []
    for xb, yb in val_ds:
        mu_p, vol_p, yz_p, var_p, cvar_p = trans_model(xb, training=False)
        vL = (
            weighted_loss(yb["log_return"], mu_p,  "log_return") +
            weighted_loss(yb["log_volume"], vol_p, "log_volume") +
            weighted_loss(yb["yz_vol"],     yz_p,  "yz_vol")    +
            weighted_loss(yb["VaR_5"],      var_p, "VaR_5")     +
            weighted_loss(yb["CVaR_5"],     cvar_p,"CVaR_5")
        )
        val_losses.append(float(vL))
    val_loss = np.mean(val_losses)

    print(f"Epoch {epoch+1}/{epochs}  val_loss: {val_loss:.6f}")

    # --- Manual early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trans_model.save_weights("best_trans_weights.weights.h5")
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Restore best weights
trans_model.load_weights("best_trans_weights.weights.h5")
print("Restored best weights.")


Epoch 1/50
Epoch loss: 0.0730869323015213
Epoch 2/50
Epoch loss: 0.06388983130455017
Epoch 3/50
Epoch loss: 0.06462130695581436
Epoch 4/50
Epoch loss: 0.05370176583528519
Epoch 5/50
Epoch loss: 0.043552808463573456
Epoch 6/50
Epoch loss: 0.03919164091348648
Epoch 7/50
Epoch loss: 0.03352444991469383
Epoch 8/50
Epoch loss: 0.029361773282289505
Epoch 9/50
Epoch loss: 0.024205373600125313
Epoch 10/50
Epoch loss: 0.020395513623952866
Epoch 11/50
Epoch loss: 0.017970740795135498
Epoch 12/50
Epoch loss: 0.014728410169482231
Epoch 13/50
Epoch loss: 0.012130418792366982
Epoch 14/50
Epoch loss: 0.010990486480295658
Epoch 15/50
Epoch loss: 0.0075329565443098545
Epoch 16/50
Epoch loss: 0.005556457210332155
Epoch 17/50
Epoch loss: 0.003984368406236172
Epoch 18/50
Epoch loss: 0.003009858075529337
Epoch 19/50
Epoch loss: 0.0014606674667447805
Epoch 20/50
Epoch loss: 0.0010596462525427341
Epoch 21/50
Epoch loss: 0.0015247559640556574
Epoch 22/50
Epoch loss: 0.0002496953238733113
Epoch 23/50
Epoch los

This code tests the Transformer model on the validation set by first predicting all five outputs. It then calculates MAE for log returns and RMSE for log volume to measure prediction accuracy. For Value at Risk (VaR), it computes the coverage, i.e., the fraction of actual returns that fall below the predicted VaR threshold, which should ideally match the expected 5% quantile. This evaluation helps you understand how well the model is predicting both standard financial metrics and risk measures.

In [ ]:
preds = trans_model.predict(X_test)
mu_p, vol_p, yz_p, var_p, cvar_p = preds

def rmse(a,b): return np.sqrt(mean_squared_error(a,b))

print("MAE log_return:", mean_absolute_error(y_test_r, mu_p))
print("RMSE log_volume:", rmse(y_test_v, vol_p))

returns = y_test_r
var_vals = var_p.reshape(-1)
coverage = np.mean(returns < var_vals)

print("VaR Coverage (5%):", coverage)

30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step
MAE log_return: 0.027177161310888657
RMSE log_volume: 0.19993214420148725
VaR Coverage (5%): 0.07407407407407407


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_all(y_true, y_pred, name):
    print(f"{name} -> MAE: {mean_absolute_error(y_true,y_pred):.5f}, RMSE: {np.sqrt(mean_squared_error(y_true,y_pred)):.5f}")

evaluate_all(y_test_r,   mu_p,  "Log Return")
evaluate_all(y_test_v,   vol_p, "Log Volume")
evaluate_all(y_test_yz,  yz_p,  "YZ Volatility")
evaluate_all(y_test_var, var_p, "VaR")
evaluate_all(y_test_cvar,cvar_p,"CVaR")

Log Return -> MAE: 0.02718, RMSE: 0.03248
Log Volume -> MAE: 0.13956, RMSE: 0.19993
YZ Volatility -> MAE: 0.03423, RMSE: 0.03646
VaR -> MAE: 0.01879, RMSE: 0.02010
CVaR -> MAE: 0.01954, RMSE: 0.02441


In [ ]:
returns = y_test_r.flatten()
VaR     = var_p.flatten()

coverage = np.mean(returns < VaR)
print("VaR Coverage (expected=5%):", coverage)

# Kupiec test – statistical validity
import numpy as np
from scipy.stats import chi2

T = len(VaR)
N = np.sum(returns < VaR)
p = 0.05

likelihood_ratio = -2*np.log(((1-p)**(T-N) * p**N) /
                             ((1-N/T)**(T-N) * (N/T)**N))

p_value = 1-chi2.cdf(likelihood_ratio,1)
print("Kupiec p-value:",p_value)

VaR Coverage (expected=5%): 0.07407407407407407
Kupiec p-value: 0.0014767471079637495


In [ ]:
CVaR_pred = cvar_p.flatten()
CVaR_true = y_test_cvar.flatten()

print("CVaR Error:", mean_absolute_error(CVaR_true, CVaR_pred))

CVaR Error: 0.019542272871449082
